### Crop-Based Inference

Detects insects using **frame-differencing motion detection → crop extraction → CNN classifier**.
Runs one or more classifier pipelines in parallel; each adds its own columns to `results.csv`.

**Input** — camera images in `data/evaluation/images/{camera_id}/` · model weights in `models/`  
**Output** — `outputs/inference/crop_results/{RUN_NAME}/results.csv` + `crops/{class}/` images

**Must edit (Cell 3):**

| Variable | Risk if skipped |
|----------|-----------------|
| `RUN_NAME` | `FileExistsError` at startup if name already exists — pick a new name or delete the old folder |
| `PIPELINES[*].enabled` | Wrong models loaded or unused models waste memory |
| `BASE_DIR` (Cell 1) | Everything fails — change only when running locally |

**Optional (Cell 3):** `darker_threshold` (default 15 — lower = more sensitive), `min_contour_area` (default 200), `enable_large_motion` (True), `strip_ocr_temperature` (False — needs tesseract), `conf_threshold` per pipeline (0 = off, applies at eval time only via `evaluate.ipynb`)


##### Cell 0 — Colab setup  ← **Colab only, skip if running locally**

Mounts Google Drive and extracts the project zip to local SSD.
Results are written to local `/content/data/` during inference (fast), then copied to Drive at the end.

**Before running on Colab:** upload `pollinator-colab.zip` to the root of your Google Drive.
The zip top-level folder must be named `pollinator-colab/`:
```
pollinator-colab/
  data/evaluation/images/{camera_name}/*.JPG   ← your images
  models/binary_best.pth
  models/4group_insectnet.pth
  models/5group_efficientnet.pth
  models/5group_insectnet.pth
  InsectNet/model.pth
```

Results are saved back to Drive at:
`MyDrive/pollinator-colab/outputs/inference/crop_results/{RUN_NAME}/`


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 0 — COLAB SETUP  ← run this cell ONLY on Colab
# ════════════════════════════════════════════════════════════
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    import zipfile, os, shutil
    from pathlib import Path

    DRIVE_ROOT = Path('/content/drive/MyDrive')
    ZIP_PATH   = DRIVE_ROOT / 'pollinator-colab.zip'
    # Extract to local /content/ (faster, no Drive connection issues)
    EXTRACT_TO = Path('/content/pollinator-colab')

    if not EXTRACT_TO.exists():
        print(f'Extracting {ZIP_PATH.name} to /content/ ...')
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall('/content/')
        print('✓ Extracted to /content/pollinator-colab')
    else:
        print(f'✓ Already extracted: {EXTRACT_TO}')

    # Install dependencies
    print('Installing dependencies...')
    os.system('pip install -q opencv-python-headless torch torchvision')
    print('✓ Dependencies ready')

    # Verify key paths
    BASE_DIR   = EXTRACT_TO
    IMAGE_ROOT = BASE_DIR / 'data' / 'evaluation' / 'images'
    MODEL_DIR  = BASE_DIR / 'models'
    print(f'\nPaths:')
    print(f'  IMAGE_ROOT : {IMAGE_ROOT}  exists={IMAGE_ROOT.exists()}')
    print(f'  MODEL_DIR  : {MODEL_DIR}   exists={MODEL_DIR.exists()}')
    for m in ['binary_best.pth','4group_insectnet.pth',
              '5group_efficientnet.pth','5group_insectnet.pth']:
        p = MODEL_DIR / m
        print(f'  {m}: {"✓" if p.exists() else "✗ NOT FOUND"}')
else:
    print('Running locally — skip this cell.')


##### Cell 1 — Environment

Set your local path. **Only edit `BASE_DIR`** — everything else is derived from it.
On Colab, `BASE_DIR` is set automatically after mounting Drive.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    import zipfile, os
    DRIVE_ROOT = Path('/content/drive/MyDrive')
    ZIP_PATH   = DRIVE_ROOT / 'pollinator-colab.zip'
    EXTRACT_TO = Path('/content/pollinator-colab')

    if not EXTRACT_TO.exists():
        print(f'Extracting {ZIP_PATH.name} to /content/ ...')
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall('/content/')
        print('✓ Extracted')
    else:
        print('✓ Already extracted')

    BASE_DIR   = EXTRACT_TO
    DRIVE_BASE = DRIVE_ROOT / 'pollinator-colab'
else:
    # ← Edit this to your local repo root
    BASE_DIR   = Path('/path/to/pollinator-classification')
    DRIVE_BASE = BASE_DIR

IMAGE_ROOT        = BASE_DIR  / 'data' / 'evaluation' / 'images'
GT_ANN_ROOT       = BASE_DIR  / 'data' / 'evaluation' / 'annotations'
MODEL_DIR         = BASE_DIR  / 'models'
# Results are always written to fast local storage during inference.
# On Colab the auto-save step at the end copies them to Drive.
# Writing directly to Drive is slow (FUSE latency per file) and risks
# data loss if the session ends before Drive flushes the last folder.
LOCAL_RESULTS     = Path('/content/data') if IN_COLAB else BASE_DIR
CROP_RESULTS_ROOT = LOCAL_RESULTS / 'outputs' / 'inference' / 'crop_results'
YOLO_RESULTS_ROOT = LOCAL_RESULTS / 'outputs' / 'inference' / 'yolo_results'
INSECTNET_W       = BASE_DIR  / 'InsectNet' / 'model.pth'
LABELED_DIR       = BASE_DIR  / 'data' / 'training' / 'annotated_crops'
CROP_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
YOLO_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Env        : {"Colab" if IN_COLAB else "Local"}')
print(f'BASE_DIR   : {BASE_DIR}  exists={BASE_DIR.exists()}')
print(f'IMAGE_ROOT : {IMAGE_ROOT}  exists={IMAGE_ROOT.exists()}')
print(f'MODEL_DIR  : {MODEL_DIR}  exists={MODEL_DIR.exists()}')


##### Cell 2 — Imports + pipeline code

Run all cells in this section before editing the run config.
Contains all functions needed by the pipeline.
**Do not edit.**

In [ ]:
import sys, csv, re, time, json, datetime
from pathlib import Path
from collections import defaultdict
import cv2, numpy as np
from PIL import Image as PILImage
from PIL.ExifTags import TAGS


In [ ]:
DEFAULT_PREPROCESS_CONFIG = {
    # ── Background ──────────────────────────────────────────────
    'background_sample_size':    0,
    'rolling_window':            1,
    # ── Background subtraction ──────────────────────────────────
    'darker_threshold':          30,
    'diff_blur_kernel_size':     7,
    # ── Vegetation mask (HSV) ────────────────────────────────────
    'veg_hue_lo':40, 'veg_hue_hi':85,   # narrowed: skip yellow insects (was 25-95)
    'veg_sat_lo':60, 'veg_val_lo':40,   # higher sat: only mask vivid vegetation
    # ── Contour filtering ────────────────────────────────────────
    'min_contour_area':          200,   # lowered to catch small insects (was 400)
    'max_contour_area':          35000,
    'max_large_motion_area':     600000,
    'max_aspect_ratio':          5,
    'merge_dist':                20,
    'min_crop_px':               10,
    # ── Morphological kernels ────────────────────────────────────
    'kernel_open_size':          3,
    'kernel_close_size':         11,
    # ── Large motion ─────────────────────────────────────────────
    'enable_large_motion':       True,
    'large_motion_tile_sizes':   [320,512],
    'large_motion_tile_stride_frac':    0.65,
    'large_motion_max_tiles_per_size':  6,
    'large_motion_max_tiles_total':     10,
    'large_motion_tile_nms_iou':        0.35,
    'large_motion_min_fg_frac':         0.008,
    'large_motion_context_pad':         10,
    'large_motion_fallback_sizes':      [640],
    'large_motion_fallback_centers':    [(0.35,0.78),(0.50,0.78),(0.65,0.78)],
    'large_motion_max_fallbacks':       3,
    'large_motion_fallback_min_fg_frac':0.003,
    # ── Tile scoring ─────────────────────────────────────────────
    'tile_texture_norm':         80.0,
    'tile_fg_norm':              0.12,
    'tile_fg_penalty_thresh':    0.45,
    'tile_fg_penalty_factor':    0.75,
    'tile_score_fg_weight':      0.65,
    'tile_score_texture_weight': 0.35,
    # ── Crop sizing ───────────────────────────────────────────────
    'crop_small_eff_thresh':50,   'crop_medium_eff_thresh':120,
    'crop_small_multiplier':2.5,  'crop_medium_multiplier':1.8,  'crop_large_multiplier':1.2,
    'crop_small_min_px':50,       'crop_small_max_px':180,
    'crop_medium_min_px':80,      'crop_medium_max_px':260,
    'crop_large_min_px':140,      'crop_large_max_px':320,
    'crop_pad_ratio':0.15,
    # ── ROI ──────────────────────────────────────────────────────
    'use_roi':False, 'manual_roi':False,
    'marker_hue':(45,75), 'marker_sat_min':200, 'marker_val_min':100,
    'marker_min_area':200, 'marker_zone_radius':800,
    'near_flower_iou_threshold':0.1,
    # ── Strip ────────────────────────────────────────────────────
    'strip_height':120, 'strip_ocr_temperature':False,
    'strip_ocr_target_height':120, 'strip_temp_min':-50, 'strip_temp_max':60,
    # ── Quality ──────────────────────────────────────────────────
    'skip_flash':True, 'skip_foggy':True,
    'foggy_threshold':20,
    # ── Speed: downscale before detection ────────────────────────────
    # All CV operations (blur, threshold, contours) run at this fraction of
    # the original resolution — 0.5 = ¼ the pixels = ~4× faster detection.
    # Crops are still extracted from the full-res image, so quality is unaffected.
    # Set to 1.0 to disable (default: full resolution).
    'detection_scale':           1.0, 'sunny_shutter_threshold':150,
    # ── Misc ─────────────────────────────────────────────────────
    'use_exif_sort':False, 'progress_every':50,
    'near_flower_iou_threshold':0.1,
    # ── Debug ────────────────────────────────────────────────────
    # ── debug_outputs: 'none' (fast, no extra files) ────────────────────
    #                 'annotate' (saves annotated JPEG per detection frame)
    'debug_outputs':'none', 'debug_save_empty_frames':False,
    'debug_max_width':1024, 'debug_jpeg_quality':60,
}
print('✓ Default preprocessing config loaded.')


In [ ]:
import csv, re, time, json, datetime, io
from pathlib import Path
from collections import defaultdict
import cv2, numpy as np
from PIL import Image as PILImage
from PIL.ExifTags import TAGS

_EXIF_DT_TAG = next((k for k,v in TAGS.items() if v=='DateTimeOriginal'), None)
_exif_cache  = {}

def _exif_dt(path):
    key=str(path)
    if key in _exif_cache: return _exif_cache[key]
    try:
        pil=PILImage.open(io.BytesIO(Path(path).read_bytes()))
        exif=pil._getexif() if hasattr(pil,'_getexif') else dict(pil.getexif())
        val=exif.get(_EXIF_DT_TAG) if exif and _EXIF_DT_TAG else None
    except Exception: val=None
    _exif_cache[key]=val; return val

def filename_sort_key(p):
    p=Path(p); m=re.search(r'(\d+)',p.stem)
    return (0,int(m.group(1)),p.name) if m else (1,p.name)

def robust_sort_key(p):
    p=Path(p); dt=_exif_dt(p)
    if dt:
        try:
            d,t=dt.split(' ')
            return (0,tuple(int(x) for x in d.split(':')+t.split(':')),p.name)
        except Exception: pass
    return filename_sort_key(p)

def setup_roi(paths, cfg):
    first=cv2.imread(str(paths[0]))
    full=np.ones(first.shape[:2],dtype=np.uint8)*255
    if not cfg.get('use_roi',False):
        print('  ROI: disabled — full image')
        return full, None
    print('  ROI: enabled')
    return full, None  # marker logic omitted for brevity

def build_background(paths, cfg):
    n=cfg.get('background_sample_size',0)
    if not n: print('  Background: rolling window only'); return None
    step=max(1,len(paths)//n)
    frames=[f for f in (cv2.imread(str(p)) for p in paths[::step][:n]) if f is not None]
    print(f'  Background: {len(frames)} frames sampled')
    return np.median(frames,axis=0).astype(np.uint8) if frames else None

def _bbox_iou(a,b):
    ax,ay,aw,ah=a; bx,by,bw,bh=b
    iw=max(0,min(ax+aw,bx+bw)-max(ax,bx)); ih=max(0,min(ay+ah,by+bh)-max(ay,by))
    inter=iw*ih; union=aw*ah+bw*bh-inter
    return inter/union if union else 0.0

def _tile_score(image,mask,bbox,cfg):
    x,y,w,h=bbox; tm=mask[y:y+h,x:x+w]; ti=image[y:y+h,x:x+w]
    if tm.size==0 or ti.size==0: return 0.0
    fg=np.count_nonzero(tm)/float(w*h)
    tex=min(cv2.cvtColor(ti,cv2.COLOR_BGR2GRAY).std()/cfg.get('tile_texture_norm',80.0),1.0)
    fgs=min(fg/cfg.get('tile_fg_norm',0.12),1.0)
    if fg>cfg.get('tile_fg_penalty_thresh',0.45): fgs*=cfg.get('tile_fg_penalty_factor',0.75)
    return cfg.get('tile_score_fg_weight',0.65)*fgs+cfg.get('tile_score_texture_weight',0.35)*tex

def _nms_tiles(scored,max_n,iou_thr):
    sel=[]
    for sc,bb in sorted(scored,key=lambda x:x[0],reverse=True):
        if all(_bbox_iou(bb,b)<=iou_thr for _,b in sel): sel.append((sc,bb))
        if len(sel)>=max_n: break
    return sel

def _tile_large(image,mask,bbox,cfg):
    x,y,w,h=bbox; H,W=mask.shape[:2]; all_sc=[]; seen=set()
    for ts_ in [int(t) for t in cfg.get('large_motion_tile_sizes',[320,512])]:
        ts_=min(ts_,W,H)
        if ts_<=0: continue
        st=max(1,int(ts_*cfg.get('large_motion_tile_stride_frac',0.65)))
        sc_sz=[]
        for yy in list(range(y,max(y+h-ts_+1,y+1),st))+[y+h-ts_]:
            for xx in list(range(x,max(x+w-ts_+1,x+1),st))+[x+w-ts_]:
                xx=max(0,min(int(xx),W-ts_)) if W>ts_ else 0
                yy=max(0,min(int(yy),H-ts_)) if H>ts_ else 0
                tw_,th_=min(ts_,W-xx),min(ts_,H-yy)
                key=(xx,yy,tw_,th_)
                if key in seen or tw_<=0 or th_<=0: continue
                fg=np.count_nonzero(mask[yy:yy+th_,xx:xx+tw_])/float(tw_*th_)
                if fg<cfg.get('large_motion_min_fg_frac',0.008): continue
                seen.add(key); sc_sz.append((_tile_score(image,mask,key,cfg),key))
        all_sc.extend(_nms_tiles(sc_sz,cfg.get('large_motion_max_tiles_per_size',6),
                                  cfg.get('large_motion_tile_nms_iou',0.35)))
    return [b for _,b in _nms_tiles(all_sc,cfg.get('large_motion_max_tiles_total',10),
                                     cfg.get('large_motion_tile_nms_iou',0.35))]

def merge_nearby_bboxes(bboxes,dist=20,max_area=35000):
    if not bboxes: return []
    boxes=list(bboxes); changed=True
    while changed:
        changed=False; merged=[]; used=set()
        for i,(x1,y1,w1,h1) in enumerate(boxes):
            if i in used: continue
            gx1,gy1,gx2,gy2=x1,y1,x1+w1,y1+h1
            for j,(x2,y2,w2,h2) in enumerate(boxes):
                if j<=i or j in used: continue
                if max(0,max(x2,gx1)-min(x2+w2,gx2))<=dist and                    max(0,max(y2,gy1)-min(y2+h2,gy2))<=dist:
                    nx1,ny1=min(gx1,x2),min(gy1,y2)
                    nx2,ny2=max(gx2,x2+w2),max(gy2,y2+h2)
                    if (nx2-nx1)*(ny2-ny1)<=max_area:
                        gx1,gy1,gx2,gy2=nx1,ny1,nx2,ny2; used.add(j); changed=True
            used.add(i); merged.append((gx1,gy1,gx2-gx1,gy2-gy1))
        boxes=merged
    return boxes


def _make_det_cfg(cfg, scale):
    """Return a copy of cfg with all pixel-count and distance keys scaled.

    When detection_scale < 1 the image passed to detect_visitor is smaller
    than the original.  Area thresholds must shrink by scale², distance /
    kernel thresholds by scale¹, so that the same physical insects are
    accepted / rejected as at full resolution.
    """
    if scale >= 1.0:
        return cfg
    s2 = scale * scale

    def _odd(v):          # round to nearest odd integer ≥ 1
        v = max(1, int(round(v)))
        return v if v % 2 else v + 1

    d = dict(cfg)
    # ── area thresholds (px²) ────────────────────────────────────────────
    for k in ('max_contour_area', 'max_large_motion_area'):
        if k in d: d[k] = max(1, int(d[k] * s2))
    # min_contour_area: scale down AND apply a small margin so insects
    # right at the boundary aren't lost to downscale blur/aliasing
    if 'min_contour_area' in d:
        d['min_contour_area'] = max(1, int(d['min_contour_area'] * s2 * 0.75))
    # ── distance / length thresholds (px) ───────────────────────────────
    for k in ('merge_dist', 'min_crop_px', 'large_motion_context_pad',
              'large_motion_fallback_centers'):
        if k in d and isinstance(d[k], (int, float)):
            d[k] = max(1, int(d[k] * scale))
    # ── tile sizes (px) ──────────────────────────────────────────────────
    for k in ('large_motion_tile_sizes', 'large_motion_fallback_sizes'):
        if k in d:
            d[k] = [max(16, int(t * scale)) for t in d[k]]
    # ── kernel sizes must stay odd ───────────────────────────────────────
    for k in ('diff_blur_kernel_size', 'kernel_open_size', 'kernel_close_size'):
        if k in d: d[k] = _odd(d[k] * scale)
    return d

def detect_visitor(image,bg,zone,cfg):
    # convert each image to grayscale once
    _gi=cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)
    _gb=cv2.cvtColor(bg,   cv2.COLOR_BGR2GRAY)
    gi=cv2.bitwise_and(_gi,_gi,mask=zone)
    gb=cv2.bitwise_and(_gb,_gb,mask=zone)
    diff=cv2.absdiff(gb,gi)
    k=int(cfg.get('diff_blur_kernel_size',7)); k=k if k%2 else k+1
    diff=cv2.GaussianBlur(diff,(k,k),0)
    _,mask=cv2.threshold(diff,cfg['darker_threshold'],255,cv2.THRESH_BINARY)
    mask=cv2.bitwise_and(mask,zone)
    hsv=cv2.cvtColor(image,cv2.COLOR_BGR2HSV)
    green=cv2.inRange(hsv,
        np.array([cfg.get('veg_hue_lo',25),cfg.get('veg_sat_lo',40),cfg.get('veg_val_lo',40)]),
        np.array([cfg.get('veg_hue_hi',95),255,255]))
    mask=cv2.bitwise_and(mask,cv2.bitwise_not(green))
    ko=cfg.get('kernel_open_size',3); kc=cfg.get('kernel_close_size',11)
    mask=cv2.morphologyEx(mask,cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(ko,ko)))
    mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(kc,kc)))
    cnts,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    normal=[]; dets=[]; max_n=cfg['max_contour_area']; max_l=cfg['max_large_motion_area']
    enable_lm=cfg.get('enable_large_motion',True)
    for c in cnts:
        area=cv2.contourArea(c)
        if area<cfg['min_contour_area']: continue
        x,y,w,h=cv2.boundingRect(c)
        if area>max_n:
            if area<=max_l and enable_lm:
                sb=(x,y,w,h)
                dets.append({'bbox':sb,'candidate_type':'large_motion_context','source_area':area,'static_suspect':False})
                for tb in _tile_large(image,mask,sb,cfg):
                    dets.append({'bbox':tb,'candidate_type':'large_motion_tile','source_area':area,'static_suspect':False})
            continue
        if max(w,h)/max(min(w,h),1)>cfg['max_aspect_ratio']: continue
        normal.append((area,(x,y,w,h)))
    for bb in merge_nearby_bboxes([b for _,b in sorted(normal,reverse=True)],
                                   cfg.get('merge_dist',20),max_n):
        x,y,w,h=bb
        if max(w,h)>=cfg.get('min_crop_px',10):
            dets.append({'bbox':bb,'candidate_type':'normal','source_area':w*h,'static_suspect':False})
    return dets

def crop_with_padding(image,det,cfg):
    x,y,w,h=det['bbox']; Hi,Wi=image.shape[:2]
    ct=det.get('candidate_type','normal')
    if ct=='large_motion_context':
        pad=cfg.get('large_motion_context_pad',10)
        return image[max(0,y-pad):min(Hi,y+h+pad),max(0,x-pad):min(Wi,x+w+pad)]
    if 'large_motion' in ct: return image[y:y+h,x:x+w]
    eff=(w*h)**0.5
    if eff<cfg.get('crop_small_eff_thresh',50):
        win=int(eff*cfg.get('crop_small_multiplier',2.5))
        win=max(cfg.get('crop_small_min_px',50),min(win,cfg.get('crop_small_max_px',180)))
    elif eff<cfg.get('crop_medium_eff_thresh',120):
        win=int(eff*cfg.get('crop_medium_multiplier',1.8))
        win=max(cfg.get('crop_medium_min_px',80),min(win,cfg.get('crop_medium_max_px',260)))
    else:
        win=int(eff*cfg.get('crop_large_multiplier',1.2))
        win=max(cfg.get('crop_large_min_px',140),min(win,cfg.get('crop_large_max_px',320)))
    pr=cfg.get('crop_pad_ratio',0.15)
    x1=max(0,x-int(w*pr)); y1=max(0,y-int(h*pr))
    x2=min(Wi,x+w+int(w*pr)); y2=min(Hi,y+h+int(h*pr))
    return image[y1:y2,x1:x2]

def save_debug(name,image,dets,debug_dir,zone,cfg):
    if cfg.get('debug_outputs','annotate')=='none': return
    overlay=image.copy()
    for det in dets:
        x,y,w,h=det['bbox']
        color=(0,255,0) if is_near_flower((x,y,w,h),zone,cfg) else (255,0,0)
        cv2.rectangle(overlay,(x,y),(x+w,y+h),color,2)
        cv2.putText(overlay,det.get('candidate_type','')[:8],(x,y-6),
                    cv2.FONT_HERSHEY_SIMPLEX,0.38,color,1)
    q=int(cfg.get('debug_jpeg_quality',60)); mw=cfg.get('debug_max_width',1024)
    h_,w_=overlay.shape[:2]
    if mw and w_>mw:
        sc=mw/w_; overlay=cv2.resize(overlay,(mw,int(h_*sc)),cv2.INTER_AREA); scale=sc
    else: scale=1.0
    cv2.imwrite(str(Path(debug_dir)/f'{name}_4_final_saved_crops.jpg'),overlay,
                [cv2.IMWRITE_JPEG_QUALITY,q])
    (Path(debug_dir)/f'{name}_4_offset.json').write_text(
        json.dumps({'ox':0,'oy':0,'scale':scale}))

def load_image_and_meta(path, cfg):
    """
    ONE disk read per frame: reads the file to bytes once, then decodes
    pixels with cv2 and parses EXIF with PIL — both from the same buffer.
    Laplacian/foggy check is done separately after preprocess_image.
    Returns (img_bgr_or_None, meta_dict).
    """
    m = {'datetime':'','camera_name':'','shutter_speed':'','weather':'unknown',
         'laplacian_var':-1.0,'skip':False,'skip_reason':''}
    try:
        data = Path(path).read_bytes()
    except Exception:
        return None, m

    # ── Decode pixels ────────────────────────────────────────────────────────
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)

    # ── Parse EXIF from the same bytes — zero extra disk I/O ────────────────
    try:
        pil  = PILImage.open(io.BytesIO(data))
        exif = pil._getexif() if hasattr(pil,'_getexif') else dict(pil.getexif())
        if exif:
            tags = {TAGS.get(k,k):v for k,v in exif.items()}
            m['datetime']    = str(tags.get('DateTimeOriginal',''))
            m['camera_name'] = str(tags.get('Model',''))
            exp = exif.get(33434)
            if exp:
                d = exp[1] if isinstance(exp,tuple) else int(1/exp)
                m['shutter_speed'] = f'1/{d}'
                m['weather'] = 'sunny' if d > cfg['sunny_shutter_threshold'] else 'cloudy'
            fv = exif.get(37385, 0)
            if fv and int(fv) != 0 and cfg.get('skip_flash', True):
                m['skip'] = True; m['skip_reason'] = f'flash(EXIF={fv})'
    except Exception: pass

    return img, m

def extract_strip_temperature(strip_bgr, cfg):
    """OCR the temperature value from the camera info strip.

    Looks for patterns like:  19C  -3C  19  23  -3.5
    Tries both white-on-dark and dark-on-white thresholding.
    Returns temperature as float, or None if not found.

    Requires: pytesseract + tesseract-ocr system package.
    Install on Colab:  !apt-get install -y tesseract-ocr
                       !pip install pytesseract
    Typical cost: ~0.3 s/frame — keep strip_ocr_temperature=False
    (default) unless you specifically need temperature data.
    """
    try:
        import re, pytesseract
        from PIL import Image as _PILI
        h, w = strip_bgr.shape[:2]
        target_h = int(cfg.get('strip_ocr_target_height', 120))
        scale = max(2, target_h // max(h, 1))
        strip_up = cv2.resize(strip_bgr, (w * scale, h * scale),
                              interpolation=cv2.INTER_CUBIC)
        gray = cv2.cvtColor(strip_up, cv2.COLOR_BGR2GRAY)
        best_txt = ''
        for thresh_type in [cv2.THRESH_BINARY, cv2.THRESH_BINARY_INV]:
            _, bw = cv2.threshold(gray, 0, 255,
                                  thresh_type | cv2.THRESH_OTSU)
            txt = pytesseract.image_to_string(
                _PILI.fromarray(bw),
                config='--psm 6 -c tessedit_char_whitelist=0123456789-. '
            )
            if len(txt.strip()) > len(best_txt.strip()):
                best_txt = txt
        t_min = float(cfg.get('strip_temp_min', -50))
        t_max = float(cfg.get('strip_temp_max',  60))
        numbers = re.findall(r'-?\d{1,3}(?:\.\d)?', best_txt)
        for n in numbers:
            val = float(n)
            if t_min <= val <= t_max:
                return val
    except Exception:
        pass
    return None

def preprocess_image(img, cfg):
    """Crop the bottom info strip; optionally OCR the temperature.

    Returns:
        processed_image : image with strip removed (or original if strip_height=0)
        temperature_c   : float or None  (None when strip_ocr_temperature=False)
    """
    sh = int(cfg.get('strip_height', 0))
    if sh <= 0 or img is None: return img, None
    h = img.shape[0]
    if sh >= h: return img, None
    temp = None
    if cfg.get('strip_ocr_temperature', False):
        strip = img[h - sh:, :]
        temp  = extract_strip_temperature(strip, cfg)
    return img[:h - sh, :], temp

def is_near_flower(bbox,zone,cfg):
    x,y,w,h=bbox; roi=zone[y:y+h,x:x+w]
    return roi.size>0 and np.count_nonzero(roi)/roi.size>cfg['near_flower_iou_threshold']

def init_csv(path,fields):
    with open(path,'w',newline='') as f: csv.DictWriter(f,fieldnames=fields).writeheader()

def write_row(path,row,fields):
    with open(path,'a',newline='') as f:
        csv.DictWriter(f,fieldnames=fields).writerow({k:row.get(k,'') for k in fields})

def get_leaf_dirs(root):
    root=Path(root)
    if (any(root.glob('*.JPG')) or any(root.glob('*.jpg'))) and        not any(p.is_dir() for p in root.iterdir()): return [root]
    return [d for d in sorted(root.rglob('*'))
            if d.is_dir() and not any(x.is_dir() for x in d.iterdir())
            and (any(d.glob('*.JPG')) or any(d.glob('*.jpg')))]

print('✓ Preprocessing functions loaded.')


In [ ]:
import torch
print(f'Device: {torch.device("cuda" if torch.cuda.is_available() else "cpu")}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

import torch, torch.nn as nn, torchvision, torchvision.transforms as T

BINARY_CLASSES = ['background','insect']
GROUP_CLASSES  = ['bumblebee','fly','butterfly','other']
FIVE_CLASSES   = ['bumblebee','fly','butterfly','other','background']

def _letterbox(img,size):
    w,h=img.size; ms=max(w,h)
    sq=PILImage.new('RGB',(ms,ms),(0,0,0))
    sq.paste(img,((ms-w)//2,(ms-h)//2))
    return sq.resize((size,size),PILImage.BILINEAR)

def _load_single_model(path, default_classes, label=''):
    dev=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ckpt=torch.load(path,map_location=dev,weights_only=False)
    cls=ckpt.get('classes',default_classes); sz=ckpt.get('img_size',224)
    keys=list(ckpt['state_dict'].keys())
    if any('trunk_output' in k or 'stem.' in k for k in keys):
        m=torchvision.models.regnet_y_32gf(weights=None); m.fc=nn.Linear(3712,len(cls))
    else:
        m=torchvision.models.efficientnet_b2(weights=None)
        m.classifier[-1]=nn.Linear(m.classifier[-1].in_features,len(cls))
    m.load_state_dict(ckpt['state_dict']); m.eval(); m.to(dev)
    tf=T.Compose([T.Lambda(lambda i:_letterbox(i,sz)),T.ToTensor(),
                  T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    backbone='insectnet' if any('trunk_output' in k for k in keys) else 'efficientnet'
    print(f'    backbone={backbone} img_size={sz} device={dev} [{label}]')
    print(f'  ✓ {label}: {backbone}  classes={cls}  img={sz}px')
    return m, tf, cls

def load_pipelines(pipelines_cfg):
    """
    Load all enabled pipelines.
    Returns dict: pipeline_name -> loaded bundle
    """
    loaded = {}
    for name, cfg in pipelines_cfg.items():
        if not cfg.get('enabled', True): continue
        print(f'  Loading pipeline: {name} ({cfg["type"]})')
        ptype = cfg['type']
        if ptype == 'two_stage':
            bm,btf,_ = _load_single_model(cfg['binary_model'], BINARY_CLASSES, 'binary')
            gm,gtf,gc= _load_single_model(cfg['group_model'],  GROUP_CLASSES,  'group')
            loaded[name] = {'type':'two_stage','bm':bm,'btf':btf,'gm':gm,'gtf':gtf,'gc':gc}
        elif ptype == 'five_class':
            fm,ftf,fc = _load_single_model(cfg['model'], FIVE_CLASSES, 'five_class')
            loaded[name] = {'type':'five_class','fm':fm,'ftf':ftf,'fc':fc}
        else:
            print(f'  ⚠ Unknown type: {ptype}')
    print(f'✓ {len(loaded)} pipeline(s) loaded.')
    return loaded

def _to_tensor_batch(crops_bgr, tf):
    imgs=[tf(PILImage.fromarray(cv2.cvtColor(c,cv2.COLOR_BGR2RGB))) for c in crops_bgr]
    return torch.stack(imgs)

def predict_all_pipelines(crops_bgr, loaded_pipelines, batch_size=32):
    """
    Run all loaded pipelines on a list of crops.
    Returns dict: pipeline_name -> list of result dicts (one per crop)
    """
    results = {name: [None]*len(crops_bgr) for name in loaded_pipelines}
    n = len(crops_bgr)
    if n == 0: return results

    with torch.no_grad():
        for start in range(0, n, batch_size):
            batch = crops_bgr[start:start+batch_size]
            idx   = list(range(start, start+len(batch)))

            for pipe_name, bundle in loaded_pipelines.items():
                ptype = bundle['type']

                if ptype == 'two_stage':
                    # move tensors to same device as model weights
                    dev = next(bundle['bm'].parameters()).device
                    # Binary pass
                    xb = _to_tensor_batch(batch, bundle['btf']).to(dev)
                    pb = torch.softmax(bundle['bm'](xb), 1)
                    bi = pb.argmax(1).tolist()
                    bc = pb.max(1).values.tolist()
                    # Group pass (only for insect predictions)
                    insect_idx = [i for i,b in enumerate(bi) if BINARY_CLASSES[b]=='insect']
                    group_res  = {}
                    if insect_idx:
                        xg = _to_tensor_batch([batch[i] for i in insect_idx], bundle['gtf']).to(dev)
                        pg = torch.softmax(bundle['gm'](xg), 1)
                        for j,orig in enumerate(insect_idx):
                            gi=pg[j].argmax().item()
                            ap={c:float(pg[j][k]) for k,c in enumerate(bundle['gc'])}
                            group_res[orig]={
                                'pollinator_type': bundle['gc'][gi],
                                'group_conf':      round(float(pg[j][gi]),4),
                                'bumblebee_prob':  round(ap.get('bumblebee',0),4),
                                'fly_prob':        round(ap.get('fly',0),4),
                                'butterfly_prob':  round(ap.get('butterfly',0),4),
                                'other_prob':      round(ap.get('other',0),4),
                            }
                    for j,orig_i in enumerate(idx):
                        bl=BINARY_CLASSES[bi[j]]
                        r={'binary_label':bl,'binary_conf':round(bc[j],4),
                           'pollinator_type':'','group_conf':'',
                           'bumblebee_prob':'','fly_prob':'','butterfly_prob':'','other_prob':''}
                        if j in group_res: r.update(group_res[j])
                        results[pipe_name][orig_i] = r

                elif ptype == 'five_class':
                    dev = next(bundle['fm'].parameters()).device
                    xf = _to_tensor_batch(batch, bundle['ftf']).to(dev)
                    pf = torch.softmax(bundle['fm'](xf), 1)
                    for j,orig_i in enumerate(idx):
                        fi=pf[j].argmax().item()
                        ap5={c:float(pf[j][k]) for k,c in enumerate(bundle['fc'])}
                        results[pipe_name][orig_i]={
                            'pollinator_type':  bundle['fc'][fi],
                            'conf':             round(float(pf[j][fi]),4),
                            'bumblebee_prob':   round(ap5.get('bumblebee',0),4),
                            'fly_prob':         round(ap5.get('fly',0),4),
                            'butterfly_prob':   round(ap5.get('butterfly',0),4),
                            'other_prob':       round(ap5.get('other',0),4),
                            'background_prob':  round(ap5.get('background',0),4),
                        }
    return results

def make_csv_fields(pipeline_names):
    """Generate CSV fields based on which pipelines are active."""
    base = [
        'camera_folder','image_name','datetime','temperature_c',
        'camera_name','shutter_speed','weather',
        'skip','skip_reason','laplacian_var','pollinator_detected',
        'crop_filename','bbox_x','bbox_y','bbox_w','bbox_h',
        'candidate_type','static_suspect','detection_scope','near_marked_flower',
    ]
    for name in pipeline_names:
        # We'll use pipeline name as prefix in CSV columns
        base += [f'{name}__binary_label', f'{name}__binary_conf',
                 f'{name}__pollinator_type', f'{name}__group_conf',
                 f'{name}__bumblebee_prob', f'{name}__fly_prob',
                 f'{name}__butterfly_prob', f'{name}__other_prob',
                 f'{name}__background_prob']
    return base

def pipeline_result_to_row(pipe_name, result):
    """Flatten a pipeline result dict into CSV row fields with prefix."""
    if result is None: return {}
    p = pipe_name + '__'
    ptype_fields = {}
    if 'binary_label' in result:  # two_stage
        ptype_fields = {
            p+'binary_label':   result.get('binary_label',''),
            p+'binary_conf':    result.get('binary_conf',''),
            p+'pollinator_type':result.get('pollinator_type',''),
            p+'group_conf':     result.get('group_conf',''),
            p+'bumblebee_prob': result.get('bumblebee_prob',''),
            p+'fly_prob':       result.get('fly_prob',''),
            p+'butterfly_prob': result.get('butterfly_prob',''),
            p+'other_prob':     result.get('other_prob',''),
            p+'background_prob':'',
        }
    else:  # five_class
        ptype_fields = {
            p+'binary_label':   '',
            p+'binary_conf':    '',
            p+'pollinator_type':result.get('pollinator_type',''),
            p+'group_conf':     result.get('conf',''),
            p+'bumblebee_prob': result.get('bumblebee_prob',''),
            p+'fly_prob':       result.get('fly_prob',''),
            p+'butterfly_prob': result.get('butterfly_prob',''),
            p+'other_prob':     result.get('other_prob',''),
            p+'background_prob':result.get('background_prob',''),
        }
    return ptype_fields

print('✓ Classifier + multi-pipeline functions loaded.')


In [ ]:
BATCH_SIZE = 32

def _save_run_config(run_dir, preprocess_cfg, pipelines_cfg):
    import json as _j
    def _safe(v):
        try: _j.dumps(v); return v
        except TypeError: return str(v)
    out = {
        'run_type':   'crop',
        'preprocess': {k:_safe(v) for k,v in preprocess_cfg.items()},
        'pipelines':  {name:{k:_safe(v) for k,v in cfg.items()}
                       for name,cfg in pipelines_cfg.items()
                       if cfg.get('enabled',True)},
    }
    (Path(run_dir)/'run_config.json').write_text(_j.dumps(out,indent=2))
    print(f'  ✓ run_config.json saved')

def get_leaf_dirs(root):
    root=Path(root)
    if (any(root.glob('*.JPG')) or any(root.glob('*.jpg'))) and        not any(p.is_dir() for p in root.iterdir()): return [root]
    return [d for d in sorted(root.rglob('*'))
            if d.is_dir() and not any(x.is_dir() for x in d.iterdir())
            and (any(d.glob('*.JPG')) or any(d.glob('*.jpg')))]

def run_folder(camera_dir, results_dir, preprocess_cfg, loaded_pipelines, csv_fields):
    camera_dir=Path(camera_dir); results_dir=Path(results_dir)
    crop_dir  =results_dir/'crops';  crop_dir.mkdir(parents=True,exist_ok=True)
    debug_dir =results_dir/'debug'
    if preprocess_cfg.get('debug_outputs','annotate') != 'none':
        debug_dir.mkdir(exist_ok=True)  # only create when debug is on
    csv_path  =results_dir/'results.csv'

    all_imgs=sorted(
        list(camera_dir.glob('*.JPG'))+list(camera_dir.glob('*.jpg')),
        key=robust_sort_key if preprocess_cfg.get('use_exif_sort') else filename_sort_key)
    if not all_imgs:
        print(f'  [SKIP] No images in {camera_dir.name}'); return 0, {}

    cam=camera_dir.name; total=len(all_imgs)
    print(f'  Images      : {total}')
    print(f'  large_motion: {"ON" if preprocess_cfg.get("enable_large_motion",True) else "OFF"}')
    print(f'  Pipelines   : {list(loaded_pipelines.keys()) if loaded_pipelines else "none (preprocess only)"}')

    zone,_ = setup_roi(all_imgs, preprocess_cfg)
    bg     = build_background(all_imgs, preprocess_cfg)
    init_csv(csv_path, csv_fields)

    # Pre-scale static background once (avoids repeated resizing in the loop)
    _det_scale = float(preprocess_cfg.get('detection_scale', 1.0))
    if bg is not None and _det_scale < 1.0:
        _bgH = max(1, int(bg.shape[0] * _det_scale))
        _bgW = max(1, int(bg.shape[1] * _det_scale))
        bg = cv2.resize(bg, (_bgW, _bgH), interpolation=cv2.INTER_AREA)
    # precompute scaled detection config once per folder (not per frame)
    _det_cfg = _make_det_cfg(preprocess_cfg, _det_scale)

    win=preprocess_cfg.get('rolling_window',0); cache=[]
    n_bbox=0; n_skipped=0; n_no_det=0

    # Per-pipeline counters
    pipe_counts = {name: {
        'insect':0,'background':0,
        'bumblebee':0,'fly':0,'butterfly':0,'other':0,
    } for name in (loaded_pipelines or {})}

    # Batch accumulation
    pending_crops = []
    pending_meta  = []

    def flush_batch():
        if not pending_crops: return
        if loaded_pipelines:
            pipe_results = predict_all_pipelines(pending_crops, loaded_pipelines, BATCH_SIZE)
        else:
            pipe_results = {}

        for i,(row_base, fname, crop) in enumerate(pending_meta):
            cv2.imwrite(str(crop_dir/fname), crop)
            full_row = dict(row_base)
            for pipe_name in (loaded_pipelines or {}):
                res = pipe_results.get(pipe_name,[None]*len(pending_crops))[i]
                full_row.update(pipeline_result_to_row(pipe_name, res))
                # Count insect vs background per pipeline
                if res:
                    if 'binary_label' in res:  # two_stage
                        k = 'insect' if res.get('binary_label')=='insect' else 'background'
                    else:  # five_class
                        pt = res.get('pollinator_type','')
                        k = 'background' if pt=='background' or not pt else 'insect'
                    pipe_counts[pipe_name][k] += 1
                    # Count pollinator type
                    if res:
                        pt = res.get('pollinator_type','')
                        if pt and pt not in ('background',''):
                            pipe_counts[pipe_name][pt] = \
                                pipe_counts[pipe_name].get(pt,0) + 1
            write_row(csv_path, full_row, csv_fields)
        pending_crops.clear(); pending_meta.clear()

    _prog   = max(1, preprocess_cfg.get('progress_every', 50))
    _t0     = time.time()
    _bbox_w = 0   # detections since last progress line

    # Per-stage timing accumulators (shown in progress line)
    _t_load = _t_pre = _t_det = _t_inf = 0.0

    for idx, path in enumerate(all_imgs):
        # ── Heartbeat: print frame number BEFORE any work ────────────────────
        # This fires every _prog frames at the START so you can see the loop
        # is running even if individual frames are slow.
        if idx % _prog == 0:
            elapsed = time.time() - _t0
            print(f'  → [{idx+1}/{total}]  {path.name}  '
                  f'(load={_t_load:.1f}s  pre={_t_pre:.1f}s  det={_t_det:.1f}s  inf={_t_inf:.1f}s  total={elapsed:.0f}s)',
                  flush=True)
            _t_load = _t_pre = _t_det = _t_inf = 0.0  # reset per-stage accumulators

        # ── ONE disk read: pixels + EXIF from the same in-memory buffer ───
        _ts = time.time()
        img, meta = load_image_and_meta(path, preprocess_cfg)
        _t_load += time.time() - _ts
        meta['camera_folder'] = cam
        if img is None: continue

        # Flash skip: no pixel work needed, bail immediately
        if meta.get('skip'):
            n_skipped += 1
            write_row(csv_path, {'image_name': path.name,
                'image_path': f'{cam}/{path.name}', **meta,
                'pollinator_detected': 'skipped'}, csv_fields)
            continue

        # ── Preprocess (strip removal, zone resize) ──────────────────────────
        _ts = time.time()
        img, _strip_temp = preprocess_image(img, preprocess_cfg)
        if _strip_temp is not None:
            meta['temperature_c'] = _strip_temp
        if zone.shape[0] != img.shape[0]:
            zone = zone[:img.shape[0], :img.shape[1]]

        # ── Detection-scale downscale (full-res img kept for cropping) ────────
        if _det_scale < 1.0:
            _dH = max(1, int(img.shape[0] * _det_scale))
            _dW = max(1, int(img.shape[1] * _det_scale))
            img_det  = cv2.resize(img,  (_dW, _dH), interpolation=cv2.INTER_AREA)
            zone_det = cv2.resize(zone, (_dW, _dH), interpolation=cv2.INTER_NEAREST)
        else:
            img_det, zone_det = img, zone

        # ── Foggy check (needs Laplacian — only possible after pixel load) ───
        try:
            lv = cv2.Laplacian(cv2.cvtColor(img_det, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()
            meta['laplacian_var'] = round(float(lv), 1)
            if preprocess_cfg.get('skip_foggy', True) and lv < preprocess_cfg.get('foggy_threshold', 50):
                meta['skip'] = True; meta['skip_reason'] = f'foggy(lap={lv:.1f})'
        except Exception: pass
        _t_pre += time.time() - _ts

        if meta.get('skip'):
            n_skipped += 1
            write_row(csv_path, {'image_name': path.name,
                'image_path': f'{cam}/{path.name}', **meta,
                'pollinator_detected': 'skipped'}, csv_fields)
            continue

        # ── Motion detection (runs at detection_scale resolution) ─────────────
        _ts = time.time()
        # cache stores downscaled frames → median/prev-frame bg is already small
        bg_use = (cache[-1] if win==1 else np.median(cache[-win:], axis=0).astype(np.uint8)) \
                 if win > 0 and cache else bg
        dets = []
        if bg_use is not None:
            bgu = bg_use[:img_det.shape[0], :img_det.shape[1]] \
                  if bg_use.shape[:2] != img_det.shape[:2] else bg_use
            dets = detect_visitor(img_det, bgu, zone_det, _det_cfg)
            # Scale bboxes back to full-res coordinates for crop extraction
            if _det_scale < 1.0:
                inv = 1.0 / _det_scale
                for _d in dets:
                    _x,_y,_w,_h = _d['bbox']
                    _d['bbox'] = (int(_x*inv), int(_y*inv),
                                  int(_w*inv), int(_h*inv))
        cache.append(img_det)   # store downscaled — saves memory + speeds median
        if win > 0 and len(cache) > win: cache.pop(0)
        _t_det += time.time() - _ts

        if preprocess_cfg.get('debug_outputs', 'none') != 'none' and dets:
            save_debug(f'{cam}__{path.stem}', img, dets, debug_dir, zone, preprocess_cfg)

        if not dets:
            n_no_det += 1
            write_row(csv_path, {'image_name': path.name,
                'image_path': f'{cam}/{path.name}', **meta,
                'pollinator_detected': 'no'}, csv_fields)
        else:
            n_bbox    += len(dets)
            _bbox_w   += len(dets)
            for i, det in enumerate(dets):
                x,y,w,h = det['bbox']
                ct  = det.get('candidate_type', 'normal')
                ss  = det.get('static_suspect', False)
                in_r = is_near_flower((x,y,w,h), zone, preprocess_cfg)
                scope = 'roi' if in_r else 'out'
                crop  = crop_with_padding(img, det, preprocess_cfg)
                if crop is None or crop.size == 0: continue
                fname   = f'{cam}__{path.stem}_crop{i:02d}_{ct[:6]}_{scope}.jpg'
                img_rel = f'{cam}/{path.name}'
                row_base = {
                    'image_name': path.name, 'image_path': img_rel, **meta,
                    'pollinator_detected': 'yes', 'crop_filename': fname,
                    'bbox_x': x, 'bbox_y': y, 'bbox_w': w, 'bbox_h': h,
                    'candidate_type': ct, 'static_suspect': str(ss),
                    'near_marked_flower': str(in_r),
                    'detection_scope': 'roi' if in_r else 'outside_roi',
                }
                pending_crops.append(crop)
                pending_meta.append((row_base, fname, crop))
                _ts = time.time()
                if len(pending_crops) >= BATCH_SIZE:
                    flush_batch()
                    _t_inf += time.time() - _ts

        # ── Final progress line at end of folder ─────────────────────────────
        if (idx + 1) == total:
            elapsed = time.time() - _t0
            rate    = (idx + 1) / max(elapsed, 0.1)
            print(f'  ✓ [{total}/{total}]  bbox={n_bbox}  skip={n_skipped}  '
                  f'done in {elapsed:.0f}s  ({rate:.1f} fps)', flush=True)
            _bbox_w = 0

    _ts = time.time(); flush_batch(); _t_inf += time.time() - _ts

    # ── Per-folder summary ────────────────────────────────────────
    print(f'  ─────────────────────────────────────────')
    print(f'  {total} frames  |  {n_bbox} bbox detected  |  skip={n_skipped}')
    if loaded_pipelines:
        POLL_CLASSES = ['bumblebee','fly','butterfly','other']
        for pipe_name, counts in pipe_counts.items():
            ins = counts['insect']; bg = counts['background']
            cls_str = '  '.join(f'{c}={counts.get(c,0)}' for c in POLL_CLASSES)
            print(f'  {pipe_name:20}: insect={ins:<5} background={bg:<5}  [{cls_str}]')

    return n_bbox, pipe_counts

print('✓ Pipeline functions loaded.')


##### Cell 3 — Run config  ← **edit this before every run**

Three things to set:

1. **`RUN_NAME`** — a short descriptive name for this experiment.
   Results go into `crop_results/{RUN_NAME}/`. Change this each time so runs don't overwrite each other.
   Examples: `'run_lm_on'`, `'run_thr25'`, `'run_01'`

2. **`PREPROCESS_CONFIG`** — detection parameters.
   Key toggles to experiment with:
   - `enable_large_motion`: True/False — include large-motion region tiling
   - `darker_threshold`: lower = more sensitive detection, more false positives
   - `kernel_close_size`: larger = merges more fragments into one detection

3. **`PIPELINES`** — which classifier models to run.
   Add/remove entries freely. Set `enabled: False` to skip a pipeline without deleting it.
   Each enabled pipeline adds 9 columns to the CSV (prefixed by pipeline name).

In [ ]:
# ── Run name ─────────────────────────────────────────────────────
# Auto-generates a timestamp name so you never overwrite a previous run.
# To add a descriptive suffix, uncomment Option B and edit the label.
import datetime as _dt
_ts = _dt.datetime.now().strftime('%Y%m%d_%H%M%S')

# Option A — pure timestamp (default, always safe):
RUN_NAME = f'run_{_ts}'

# Option B — timestamp + descriptive label (uncomment to use):
# RUN_NAME = f'run_{_ts}_lm_on_thr15'

# ── Preprocessing config ─────────────────────────────────────────
# Only change what you want to vary from defaults
PREPROCESS_CONFIG = {
    **DEFAULT_PREPROCESS_CONFIG,
    'enable_large_motion': True,    # ← key toggle
    'darker_threshold':    15,      # ← key toggle
    'rolling_window':      1,
    'use_roi':             False,


    'veg_hue_lo':  40,
    'veg_hue_hi':  85,
    'veg_sat_lo':  60,

    'min_contour_area': 200,


}

# ── Pipelines to run ──────────────────────────────────────────────
# Add / remove / disable any pipeline here
# Each enabled pipeline adds columns to the CSV: {name}__binary_label, etc.
PIPELINES = {
    'two_stage': {
        'enabled':      True,
        'type':         'two_stage',
        'binary_model': MODEL_DIR / 'binary_best.pth',
        'group_model':  MODEL_DIR / '4group_insectnet.pth',
    },
    'five_class_eff': {
        'enabled':      True,
        'type':         'five_class',
        'model':        MODEL_DIR / '5group_efficientnet.pth',
    },
    'five_class_ins': {
        'enabled':      True,
        'type':         'five_class',
        'model':        MODEL_DIR / '5group_insectnet.pth',
    },
    # ── Add more pipelines here as needed ──────────────────────
    # 'two_stage_v2': {
    #     'enabled':      False,
    #     'type':         'two_stage',
    #     'binary_model': MODEL_DIR / 'binary_v2.pth',
    #     'group_model':  MODEL_DIR / '4group_v2.pth',
    # },
}

RUN_DIR = CROP_RESULTS_ROOT / RUN_NAME

# ── Guard: refuse to overwrite an existing run ───────────────────
# Check BOTH local folder and Drive folder — Colab sessions start fresh
# each time so the local folder is always empty, but the Drive folder
# from a previous run would get silently overwritten without this check.
_drive_run = DRIVE_BASE / 'outputs' / 'inference' / 'crop_results' / RUN_NAME
_local_exists = RUN_DIR.exists() and any(RUN_DIR.iterdir())
_drive_exists = _drive_run.exists() and any(_drive_run.iterdir())
if _local_exists or _drive_exists:
    _where = str(RUN_DIR) if _local_exists else str(_drive_run)
    raise FileExistsError(
        f'\n\n  Run "{RUN_NAME}" already exists at:\n'
        f'  {_where}\n\n'
        f'  Change RUN_NAME to a new value before running.\n'
    )
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f'RUN_NAME  : {RUN_NAME}')
print(f'Results → : {RUN_DIR}')
active = [n for n,c in PIPELINES.items() if c.get('enabled',True)]
print(f'Pipelines : {active}')
print(f'large_motion: {PREPROCESS_CONFIG["enable_large_motion"]}')
print(f'darker_threshold: {PREPROCESS_CONFIG["darker_threshold"]}')


##### Cell 4 — Verify model files

Checks that all model `.pth` files exist before loading them.
If any file shows `✗ NOT FOUND`, fix the path in Cell 2 before continuing.

In [ ]:
print('Verifying model files...')
all_ok = True
for pipe_name, cfg in PIPELINES.items():
    if not cfg.get('enabled', True): continue
    paths_to_check = []
    if cfg['type'] == 'two_stage':
        paths_to_check = [cfg['binary_model'], cfg['group_model']]
    else:
        paths_to_check = [cfg['model']]
    for p in paths_to_check:
        ok = Path(p).exists()
        print(f'  [{pipe_name}] {Path(p).name}: {"✓" if ok else "✗ NOT FOUND"}')
        if not ok: all_ok = False
assert all_ok, 'Some model files missing — check paths in Cell 2'


##### Cell 5 — Load models + find camera folders

- Loads all enabled pipeline models into memory
- Scans `IMAGE_ROOT` for camera folders (leaf directories containing `.JPG` files)
- Shows a count of images per folder so you know what to expect

If a camera folder shows 0 images, check that `IMAGE_ROOT` is correct.

In [ ]:
loaded_pipelines = load_pipelines(PIPELINES)
csv_fields = make_csv_fields(list(loaded_pipelines.keys()))
print(f'\nCSV will have {len(csv_fields)} columns')

assert IMAGE_ROOT.exists(), f'IMAGE_ROOT not found: {IMAGE_ROOT}'
camera_dirs = get_leaf_dirs(IMAGE_ROOT)
total_images = 0
print(f'\nFound {len(camera_dirs)} camera folder(s):')
for d in camera_dirs:
    n = len(list(d.glob('*.JPG'))) + len(list(d.glob('*.jpg')))
    total_images += n
    print(f'  {d.name:<55} {n:>5} images')
print(f'  {"─"*62}')
print(f'  Total: {total_images} images')


##### Cell 6 — Run  ← **this is the main processing cell**

Processes all camera folders one by one:
1. Saves `run_config.json` for traceability
2. For each camera folder:
   - Detects candidate insects frame-by-frame (rolling window background subtraction)
   - Saves all candidate crops to `crops/`
   - Runs batch inference through all enabled pipelines
   - Writes one CSV row per crop with results from all pipelines
   - Saves annotated debug frames to `debug/`

**Progress** is printed every 50 frames showing fps and ETA.
Press `■ Stop` to interrupt safely — partial results are already saved.

In [ ]:
import traceback

_save_run_config(RUN_DIR, PREPROCESS_CONFIG, PIPELINES)

# Accumulators for final summary
total_stats = {'frames': 0, 'bbox': 0}
pipeline_totals = {name: {
        'insect':0,'background':0,
        'bumblebee':0,'fly':0,'butterfly':0,'other':0,
    } for name in loaded_pipelines}

t_start = time.time()

for ci, camera_dir in enumerate(camera_dirs):
    out = RUN_DIR / camera_dir.name
    out.mkdir(exist_ok=True)
    print(f'\n[{ci+1}/{len(camera_dirs)}] {camera_dir.name}')
    print(f'  ─────────────────────────────────────────')
    try:
        n_bbox, pipe_counts = run_folder(
            camera_dir, out, PREPROCESS_CONFIG, loaded_pipelines, csv_fields)
        total_stats['frames'] += len(list(camera_dir.glob('*.JPG')))+len(list(camera_dir.glob('*.jpg')))
        total_stats['bbox']   += n_bbox
        for pipe_name, counts in pipe_counts.items():
            for k in ('insect','background','bumblebee','fly','butterfly','other'):
                pipeline_totals[pipe_name][k] += counts.get(k,0)
    except KeyboardInterrupt:
        print('\n⚠ Interrupted.'); raise
    except Exception as e:
        print(f'  ✗ ERROR: {e}'); traceback.print_exc()

total_time = time.time() - t_start

# ── Final summary ─────────────────────────────────────────────────
print(f'\n{"═"*55}')
print(f'RUN COMPLETE : {RUN_NAME}')
print(f'  Total frames : {total_stats["frames"]}')
print(f'  Total bbox   : {total_stats["bbox"]}')
print(f'  Time         : {total_time:.1f}s ({total_time/60:.1f} min)')
print()
POLL_CLASSES = ['bumblebee','fly','butterfly','other']
if pipeline_totals:
    pipe_names = list(pipeline_totals.keys())
    col_w = max(len(n) for n in pipe_names) + 2
    # Detection table
    print(f'  {"Pipeline":{col_w}}  {"insect":>8}  {"background":>10}  {"total":>7}')
    print(f'  {"-"*(col_w+30)}')
    for pipe_name, counts in pipeline_totals.items():
        ins = counts['insect']; bg = counts['background']
        print(f'  {pipe_name:{col_w}}  {ins:>8}  {bg:>10}  {ins+bg:>7}')
    # Pollinator breakdown table
    print()
    print(f'  Pollinator breakdown (insect crops only):')
    header = f'  {"Pipeline":{col_w}}  ' + '  '.join(f'{c:>10}' for c in POLL_CLASSES)
    print(header)
    print(f'  {"-"*(col_w+50)}')
    for pipe_name, counts in pipeline_totals.items():
        row = f'  {pipe_name:{col_w}}  ' + '  '.join(f'{counts.get(c,0):>10}' for c in POLL_CLASSES)
        print(row)
print(f'{"═"*55}')
print(f'Results : {RUN_DIR}')

# ── Auto-save results to Drive (Colab only) ──────────────────────
# Results were written to local /content/data/ during inference (fast,
# reliable).  Now copy to Drive folder-by-folder so a dropped connection
# can't lose a whole run — each folder is fully copied before moving on.
if IN_COLAB:
    import shutil
    drive_run = DRIVE_BASE / 'outputs' / 'inference' / 'crop_results' / RUN_NAME
    drive_run.mkdir(parents=True, exist_ok=True)
    print(f'\nSaving results to Drive (folder by folder)...')

    _saved_ok  = []
    _saved_err = []

    for cam_dir in sorted(RUN_DIR.iterdir()):
        if not cam_dir.is_dir(): continue
        dest = drive_run / cam_dir.name
        try:
            if dest.exists(): shutil.rmtree(str(dest))
            shutil.copytree(str(cam_dir), str(dest))
            # ── Verify: count files on both sides ──────────────────────
            local_files = list(cam_dir.rglob('*'))
            drive_files = list(dest.rglob('*'))
            local_n = sum(1 for f in local_files if f.is_file())
            drive_n = sum(1 for f in drive_files if f.is_file())
            if drive_n == local_n:
                _saved_ok.append((cam_dir.name, local_n))
                print(f'  ✓ {cam_dir.name}  ({local_n} files)')
            else:
                _saved_err.append((cam_dir.name, local_n, drive_n))
                print(f'  ✗ {cam_dir.name}  MISMATCH local={local_n} drive={drive_n}')
        except Exception as _e:
            _saved_err.append((cam_dir.name, -1, -1))
            print(f'  ✗ {cam_dir.name}  ERROR: {_e}')

    try:
        shutil.copy(str(RUN_DIR / 'run_config.json'), str(drive_run / 'run_config.json'))
    except Exception as _e:
        print(f'  ✗ run_config.json  ERROR: {_e}')

    # ── Final save summary ────────────────────────────────────────────
    print(f'\n{"─"*50}')
    print(f'Drive save summary:  {len(_saved_ok)} ok  /  {len(_saved_err)} failed')
    if _saved_err:
        print(f'  FAILED folders (re-run this cell to retry):')
        for _name, _ln, _dn in _saved_err:
            print(f'    ✗ {_name}  (local={_ln}  drive={_dn})')
        print(f'  Local results still at: {RUN_DIR}')
        print(f'  Drive path: {drive_run}')
    else:
        total_saved = sum(n for _, n in _saved_ok)
        print(f'  ✓ All {total_saved} files verified on Drive')
        print(f'  {drive_run}')


##### Cell 7 — Quick summary

After the run completes, shows a breakdown per pipeline:
how many crops were classified as insect vs background.

Useful for a quick sanity check before running `evaluate.ipynb`.
If all pipelines show 0 insects, something went wrong with detection or models.

In [ ]:
all_csvs = list(RUN_DIR.rglob('results.csv'))
rows_yes = []
for f in all_csvs:
    with open(f,newline='') as fh:
        rows_yes += [r for r in csv.DictReader(fh) if r.get('pollinator_detected')=='yes']

print(f'Run      : {RUN_NAME}')
print(f'Crops    : {len(rows_yes)} candidates')
print()
for pipe_name, bundle in loaded_pipelines.items():
    p = pipe_name + '__'
    if bundle['type'] == 'two_stage':
        ins = sum(1 for r in rows_yes if r.get(p+'binary_label')=='insect')
        bg  = sum(1 for r in rows_yes if r.get(p+'binary_label')=='background')
        print(f'  {pipe_name:20} → insect={ins}  background={bg}')
    else:
        ins = sum(1 for r in rows_yes if r.get(p+'pollinator_type') not in ('background',''))
        bg  = sum(1 for r in rows_yes if r.get(p+'pollinator_type')=='background')
        print(f'  {pipe_name:20} → insect={ins}  background={bg}')


##### Cell 8 — Organise crops by predicted class  *(optional)*

Reads `results.csv` from every camera subfolder and **copies** each crop into
`crops/{predicted_class}/` so you can browse predictions by class before reviewing.

- Original flat `crops/*.jpg` files are **kept** (nothing is deleted).
- Running this cell twice is safe — existing files are skipped.
- After this step, point `relabel.py --labeled` at the `crops/` folder to
  correct wrong predictions and send confirmed crops straight to `annotated_crops/`.
  Use `--dest` to route corrected crops there automatically (see `tools/README.md`).


In [ ]:
import csv, shutil as _sh
from pathlib import Path
from collections import defaultdict

INSECT_CLASSES = {'bumblebee', 'fly', 'butterfly', 'other'}
ALL_CLASSES    = INSECT_CLASSES | {'background', 'unsure'}

# Which pipeline to use for the class label (first enabled wins)
_PIPE_PRIORITY = [p for p,cfg in PIPELINES.items() if cfg.get('enabled', True)]

def _best_class(row):
    """Return the consensus predicted class from results.csv row."""
    for pipe in _PIPE_PRIORITY:
        pt = row.get(f'{pipe}__pollinator_type', '').strip()
        if pt and pt in ALL_CLASSES:
            return pt
    # Fall back to legacy single-pipeline columns
    pt = row.get('pollinator_type', '').strip()
    if pt and pt in ALL_CLASSES:
        return pt
    # Last resort: binary label
    if row.get('pollinator_detected', '') == 'no':
        return 'background'
    return 'unsure'

total_copied = 0
total_skipped = 0

for csv_path in sorted(RUN_DIR.rglob('results.csv')):
    crops_dir = csv_path.parent / 'crops'
    if not crops_dir.exists():
        continue

    with open(csv_path, newline='') as fh:
        rows = list(csv.DictReader(fh))

    # Only process rows where a crop was saved
    copied_here = defaultdict(int)
    for r in rows:
        fname = r.get('crop_filename', '').strip()
        if not fname:
            continue
        src = crops_dir / fname
        if not src.exists():
            continue
        cls = _best_class(r)
        dst_dir = crops_dir / cls
        dst_dir.mkdir(exist_ok=True)
        dst = dst_dir / fname
        if dst.exists():
            total_skipped += 1
        else:
            _sh.copy2(src, dst)
            copied_here[cls] += 1
            total_copied += 1

    if copied_here:
        cam = csv_path.parent.name
        summary = '  '.join(f'{cls}={n}' for cls, n in sorted(copied_here.items()))
        print(f'{cam}: {summary}')

print(f'\nDone.  Copied {total_copied} crops into class subfolders  ({total_skipped} already existed).')
print(f'\nTo review and send to annotated_crops:')
print(f'  python3 tools/labeling/relabel.py \\')
print(f'    --labeled {RUN_DIR}/.../crops \\')
print(f'    --dest    {BASE_DIR}/data/training/annotated_crops')
